<a href="https://colab.research.google.com/github/bhatturvashi2618-sketch/capstoneproject2/blob/main/Copy_of_loan_approval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Business Understanding
## Project Overview

Banks need to identify customers who are likely to default on loans before approval. Incorrect lending decisions can lead to financial losses. The goal of this project is to build a machine learning model that predicts loan default using customer and loan information, helping banks reduce risk and make better lending decisions.

## Problem Statement

Develop a machine learning model to predict whether a loan applicant is likely to default based on historical customer and loan data.

##Objective

Predict loan default.
Reduce financial losses.
Support better loan approval decisions.

![](https://raw.githubusercontent.com/bhatturvashi2618-sketch/capstoneproject2/main/loanapproval.png)



**Import Libraries**

In [ ]:
#Data Manipulation
import pandas as pd
import numpy as np

In [ ]:
#Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#Ignore Warnings
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#Data Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

In [ ]:
#Handle Missing Values
from sklearn.impute import SimpleImputer

In [ ]:
!pip install xgboost
#Machine Learning Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

In [ ]:
#Model Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    roc_curve
)

In [ ]:
#Hyperparameter Tuning
from sklearn.model_selection import RandomizedSearchCV

In [ ]:
#Cross Validation
from sklearn.model_selection import cross_val_score

In [ ]:
#Feature Importance
from sklearn.inspection import permutation_importance

In [ ]:
#Explainable AI (Optional but Recommended)
import shap

In [ ]:
#Save the Model
import joblib

#Data Source

The dataset used in this project was obtained from Kaggle. It is a publicly available Loan Approval Prediction dataset provided in CSV format.
The data was imported into Python using the Pandas library for preprocessing, exploratory data analysis, and machine learning model development.

#Constraints

The dataset contains missing values that require preprocessing.
Some categorical features must be encoded before model training.
The dataset is relatively small, so careful validation is needed to avoid overfitting.
The data is intended for educational and research purposes and may not fully represent real-world banking scenarios.

In [ ]:
# Setting this option will print all collumns of a dataframe
pd.set_option('display.max_columns', None)
# Setting this option will print all of the data in a feature
pd.set_option('display.max_colwidth', None)

In [ ]:
import pandas as pd
accepted_df = pd.read_csv('/content/sample_data/accepted_2007_to_2018Q4.csv', nrows=10000)
display(accepted_df.head())

FileNotFoundError: [Errno 2] No such file or directory: '/content/sample_data/accepted_2007_to_2018Q4.csv'

In [ ]:
import pandas as pd
rejected_df = pd.read_csv('/content/sample_data/rejected_2007_to_2018Q4.csv', nrows=10000)
display(rejected_df.head())

#Data Dictionary
The dataset contains over 140 features, including borrower demographics, employment information, loan details, credit history, repayment records, and account balances. For brevity, only the key variables used in model development are described in this report.

#Exploratory Data Analysis (EDA) & Preprocessing

In [ ]:
#Identifiers such as id, member_id, url, and desc do not help the model learn patterns. They are unique for each loan or contain unnecessary text, so they should be removed.
# Drop identifier columns
columns_to_drop = [
    'id', 'member_id', 'url', 'desc', 'title', 'emp_title',

    'out_prncp', 'out_prncp_inv',
    'total_pymnt', 'total_pymnt_inv',
    'total_rec_prncp', 'total_rec_int',
    'total_rec_late_fee', 'recoveries',
    'collection_recovery_fee',
    'last_pymnt_d', 'last_pymnt_amnt',
    'next_pymnt_d', 'last_credit_pull_d',
    'last_fico_range_high', 'last_fico_range_low',

    'hardship_flag', 'hardship_type', 'hardship_reason',
    'hardship_status', 'deferral_term',
    'hardship_amount', 'hardship_start_date',
    'hardship_end_date', 'payment_plan_start_date',
    'hardship_length', 'hardship_dpd',
    'hardship_loan_status',
    'orig_projected_additional_accrued_interest',
    'hardship_payoff_balance_amount',
    'hardship_last_payment_amount',

    'debt_settlement_flag',
    'debt_settlement_flag_date',
    'settlement_status',
    'settlement_date',
    'settlement_amount',
    'settlement_percentage',
    'settlement_term'
]

accepted_df.drop(columns=columns_to_drop, inplace=True, errors='ignore')
rejected_df.drop(columns=['Loan Title', 'Policy Code'], inplace=True, errors='ignore')

In [ ]:
print('Columns in accepted_df:')
print(accepted_df.columns.tolist())

In [ ]:
print('Columns in rejected_df:')
print(rejected_df.columns.tolist())

In [ ]:
missing = pd.DataFrame({
    'Missing Values': accepted_df.isnull().sum(),
    'Percentage': (accepted_df.isnull().sum() / len(accepted_df)) * 100
})

missing = missing[missing['Missing Values'] > 0]
missing.sort_values('Percentage', ascending=False)

In [ ]:
missing = pd.DataFrame({
    'Missing Values': rejected_df.isnull().sum(),
    'Percentage': (rejected_df.isnull().sum() / len(rejected_df)) * 100
})

missing = missing[missing['Missing Values'] > 0]
missing.sort_values('Percentage', ascending=False)

In [ ]:
#drop columns that have more than 70% of missing values
columns_to_drop = [
'sec_app_inq_last_6mths'	,
'sec_app_inq_last_6mths'	,
'sec_app_fico_range_high'	,
'sec_app_fico_range_low'	,
'sec_app_num_rev_accts'	,
'sec_app_chargeoff_within_12_mths'	,
'sec_app_collections_12_mths_ex_med'	,
'sec_app_open_act_il'	,
'sec_app_mths_since_last_major_derog'	,
'sec_app_revol_util'	,
'sec_app_open_acc'	,
'sec_app_mort_acc'	,
'revol_bal_joint'	,
'annual_inc_joint'	,
'verification_status_joint'	,
'dti_joint'	,
'mths_since_last_record'	,
'mths_since_recent_bc_dlq'	,
'mths_since_last_major_derog'


]

accepted_df.drop(columns=columns_to_drop, inplace=True, errors='ignore')


#axis=1 → Drop columns.
#thresh=threshold → Keep columns that have at least 70% non-missing values.
threshold = len(accepted_df) * 0.7
accepted_df = accepted_df.dropna(axis=1, thresh=threshold)

threshold = len(rejected_df) * 0.7
rejected_df = rejected_df.dropna(axis=1, thresh=threshold)

In [ ]:
missing = (accepted_df.isnull().sum() / len(accepted_df)) * 100
missing.sort_values(ascending=False)

In [ ]:
missing = (rejected_df.isnull().sum() / len(rejected_df)) * 100
missing.sort_values(ascending=False)

Check the Distribution of Numerical Columns

In [ ]:
columns = [
'il_util',
'mths_since_recent_inq',
'num_tl_120dpd_2m'
]

accepted_df[columns].hist(figsize=(12, 8), bins=30)
plt.tight_layout()
plt.show()
#data is skewed

In [ ]:
columns = [
'Risk_Score'
]

rejected_df[columns].hist(figsize=(12, 8), bins=30)
plt.tight_layout()
plt.show()

In [ ]:
#check for skewness
print("il_util:", accepted_df['il_util'].skew())
print("mths_since_recent_inq:", accepted_df['mths_since_recent_inq'].skew())
print("num_tl_120dpd_2m:", accepted_df['num_tl_120dpd_2m'].skew())
print("num_tl_120dpd_2m:", accepted_df['num_tl_120dpd_2m'].skew())
print("Risk_Score:", rejected_df['Risk_Score'].skew())

In [ ]:
accepted_df['il_util'] = accepted_df['il_util'].fillna(accepted_df['il_util'].median())
accepted_df['mths_since_recent_inq'] = accepted_df['mths_since_recent_inq'].fillna(accepted_df['mths_since_recent_inq'].median())
rejected_df['Risk_Score'] = rejected_df['Risk_Score'].fillna(rejected_df['Risk_Score'].median())

In [ ]:
#check for duplicates
accepted_df.duplicated().sum()

In [ ]:
rejected_df.duplicated().sum()

In [ ]:
#if exists
accepted_df.drop_duplicates(inplace=True)

In [ ]:
rejected_df.drop_duplicates(inplace=True)

In [ ]:
#identify outliers
import seaborn as sns
import matplotlib.pyplot as plt

cols = [
    'annual_inc',
    'loan_amnt',
    'revol_bal',
    'avg_cur_bal',
    'tot_hi_cred_lim'
]

for col in cols:
    plt.figure(figsize=(6,4))
    sns.boxplot(x=accepted_df[col])
    plt.title(col)
    plt.show()

In [ ]:
import seaborn as sns

sns.boxplot(x=rejected_df['Risk_Score'])
plt.show()

In [ ]:
#nvestigate whether a score of 0 is valid.
rejected_df[rejected_df['Risk_Score'] == 0]

In [ ]:
#Check whether 0 is a valid value
rejected_df['Risk_Score'].describe()


In [ ]:
import numpy as np

# Replace 0 with NaN
rejected_df['Risk_Score'] = rejected_df['Risk_Score'].replace(0, np.nan)

# Impute with median
rejected_df['Risk_Score'] = rejected_df['Risk_Score'].fillna(rejected_df['Risk_Score'].median())

In [ ]:
#verify
print(rejected_df['Risk_Score'].min())
print(rejected_df['Risk_Score'].isnull().sum())

In [ ]:
#check for the skew ness
rejected_df['Risk_Score'].skew()

In [ ]:
#shape before capping
print(accepted_df.shape)

In [ ]:
#capping the outliars
columns = [
    'revol_bal',
    'avg_cur_bal',
    'tot_hi_cred_lim'
]

for col in columns:
    Q1 = accepted_df[col].quantile(0.25)
    Q3 = accepted_df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    accepted_df[col] = accepted_df[col].clip(lower=lower, upper=upper)

In [ ]:
#shape after capping
print(accepted_df.shape)

In [ ]:
#check for catogorical data
cat_cols = accepted_df.select_dtypes(include=['object', 'category']).columns

print(cat_cols.tolist())

In [ ]:
cat_cols = rejected_df.select_dtypes(include=['object', 'category']).columns

print(cat_cols.tolist())

In [ ]:
#encoding the categorical data
rejected_df = pd.get_dummies(rejected_df, columns=['Application Date', 'Debt-To-Income Ratio', 'Zip Code', 'State', 'Employment Length'], drop_first=True)

In [ ]:
#check data types
accepted_df.dtypes


In [ ]:
rejected_df.dtypes

In [ ]:
#Separate features and target
X = accepted_df.drop('loan_status', axis=1)
y = accepted_df['loan_status']

X = rejected_df.drop('loan_status', axis=1)
y = rejected_df['loan_status']

## Connect Google Drive to Colab

To access files stored in your Google Drive, you first need to mount your Drive in your Colab notebook. This will prompt you to authenticate your Google account.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

Once your Drive is mounted, your files will be accessible under `/content/drive/My Drive/`. For example, if your CSV files are in a folder named `my_data` in your Drive, you would access them like this:

```python
accepted_df = pd.read_csv('/content/drive/My Drive/my_data/accepted_2007_to_2018Q4.csv', nrows=10000)
rejected_df = pd.read_csv('/content/drive/My Drive/my_data/rejected_2007_to_2018Q4.csv', nrows=10000)
```

---

## Connect Colab to GitHub

There are a few ways to connect Colab to GitHub:

1.  **Clone a repository**: If you want to work with a repository, you can clone it directly into your Colab environment using `!git clone`.

    ```python
    !git clone https://github.com/your-username/your-repository.git
    ```

2.  **Save a copy to GitHub**: You can save your Colab notebook directly to GitHub by going to `File` -> `Save a copy to GitHub`.

3.  **Open a notebook from GitHub**: You can open a notebook directly from GitHub by going to `File` -> `Open notebook` and then selecting the `GitHub` tab.